# E1b · Detección ciega FoV (matched filter)

**Spec:** [`docs/spec_E1b_codex_fov_detection.md`](../docs/spec_E1b_codex_fov_detection.md)  |  **Bloque:** E · Resultado  |  **Run de este set:** `ROXs12b_realigned`

Mapas matched-filter espacio-espectrales sobre todo el campo desde los cubos residuales (sgf/lpm/psfsub), normalizados por ruido de anillos; propone candidatos ciegos (Julo et al. 2025 Fig. 9 + App. G).

| | |
|---|---|
| **Entrada** | Cubos residuales C5/C6 (+psfsub reconstruido) + PSF C1 + posiciones B3 |
| **Salida (QC/productos)** | `stages/stage_h01b_qc.json`, `stage_h01b_fovmap_<m>.fits` |
| **Consume aguas abajo** | E6 (ROC); targets nuevos: candidatos previos a B3 (checkpoint usuario) |


## Qué hace E1b

Para cada cubo residual: plantilla espectral gaussiana (FWHM=LSF) en la línea objetivo → mapa M = Σ f·residual → correlación con el kernel de PSF cromática C1 → normalización por anillos (μ, σ robustos por radio, filosofía Andres 1994) → mapa z y candidatos con z ≥ 5 fuera del núcleo AO.

Es la etapa de **detección ciega** del pipeline multi-target: en targets nuevos corre antes de fijar `companion` en B3 (la promoción de un candidato es SIEMPRE checkpoint del usuario); con compañera conocida es un test de consistencia con E1 (su z se reporta como `known_source`). El QC registra además la gaussianidad del ruido por anillos (App. G del paper): no-gaussianidad cerca del núcleo NO bloquea — motiva la estadística empírica de E3/E6.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_h01b_fovmap --run-id $RUN
```

Ligero (~1 min por método).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h01b_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_h01b_fovmap --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h01b_qc.json', RUN_ID)
nb.show(qc, keys=['params.threshold_sigma', 'params.kernel_source', 'checks.v1_maps_written', 'checks.v2_known_source_reported'], title='E1b')


## Evidencia: candidatos, fuente conocida y ruido por anillos


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E1b', 'stages/stage_h01b_qc.json'):
        q = nb.load_qc('stages/stage_h01b_qc.json', RUN_ID)
        print('kernel:', q['params']['kernel_source'], ' umbral:', q['params']['threshold_sigma'], 'σ')
        for m, mq in q['methods'].items():
            ks = mq['known_source']
            z_known = None if ks is None else ks['z']
            print(f"  {m:14s} candidatos={len(mq['candidates'])}  z(compañera)={z_known}")
            for c in mq['candidates'][:5]:
                print(f"     cand y={c['y']} x={c['x']} r={c['r_px']:.1f}px z={c['z']:.1f}")
        print('checks:', q['checks'])


## Plot — mapas z por método + gaussianidad por anillos

Mapas z (Fig. 9 del paper) con estrella (*), compañera conocida (○) y candidatos (□); y fracción |z|>3 por anillo vs expectativa gaussiana (App. G).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    q = nb.load_qc('stages/stage_h01b_qc.json', RUN_ID)
    methods = list(q['methods'])
    fig, axes = plt.subplots(1, len(methods) + 1, figsize=(5*(len(methods)+1), 4.4))
    for ax, m in zip(axes, methods):
        with fits.open(q['methods'][m]['map_fits']) as h:
            z = np.asarray(h['Z'].data, float)
        im = ax.imshow(z, origin='lower', cmap='viridis', vmin=-3, vmax=8)
        sy, sx = q['star_yx']; ax.plot(sx, sy, '*', color='red')
        if q.get('companion_yx'):
            cy, cx = q['companion_yx']; ax.plot(cx, cy, 'o', mfc='none', mec='lime', ms=12)
        for c in q['methods'][m]['candidates']:
            ax.plot(c['x'], c['y'], 's', mfc='none', mec='orange', ms=10)
        ax.set_title(f'{m} · z'); ax.axis('off'); fig.colorbar(im, ax=ax, shrink=0.75)
    ax2 = axes[-1]
    for m in methods:
        rn = q['methods'][m]['ring_noise']
        r = [0.5*(x['r_lo']+x['r_hi']) for x in rn]
        f3 = [x['frac_abs_z_gt3'] for x in rn]
        ax2.plot(r, f3, marker='o', ms=3, label=m)
    ax2.axhline(0.0027, color='k', ls=':', label='gaussiana (0.27%)')
    ax2.set_xlabel('radio [px]'); ax2.set_ylabel('frac |z|>3'); ax2.set_yscale('log')
    ax2.legend(fontsize=7); ax2.set_title('gaussianidad por anillos (App. G)')
    fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Umbral 5σ y r_min=3px congelados; la promoción de candidatos a `companion` (B3) es checkpoint del usuario, nunca automática. · [`docs/spec_E1b_codex_fov_detection.md`](../docs/spec_E1b_codex_fov_detection.md)
- Normalización por anillos con μ/σ robustos (Andres 1994); no-gaussianidad registrada, no bloqueante. · [`docs/plan_integracion_halosub_julo2025.md`](../docs/plan_integracion_halosub_julo2025.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/stage_h01b_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (requiere C5/C6 corridos). Núcleo y contrato verificados con tests sintéticos (2026-07-14).
